# 4.4 动态 Shape 与控制流

这一节开始处理更接近真实工程的执行逻辑：动态 shape、循环和条件分支。

前面的章节里，大多数输入 shape 都是固定的；但推理服务里 batch size、序列长度、隐藏维度都可能变化。PyPTO 提供 `pypto.DYNAMIC`、`pypto.view`、`pypto.assemble`、`pypto.loop` 和条件语句来描述这些变化。

## 1. 环境准备

本节练习数量较多，先统一准备环境、参考实现和运行模式。

In [ ]:
import os
os.environ['TILE_FWK_DEVICE_ID'] = '0'
from dataclasses import dataclass
from typing import Optional
import torch
import pypto
import numpy as np
from numpy.testing import assert_allclose
import torch_npu


def get_device():
    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    return f"npu:{device_id}"


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<not set>"))
print("device:", device)
print("run_mode:", RUN_MODE)
print("pypto:", pypto.__file__)


## 2. 学完后你应该能够

学完这一节后，可以回到这里检查自己是否已经做到：

1. 使用 `pypto.DYNAMIC` 描述动态 batch 或动态隐藏维度。
2. 理解 `view -> compute -> assemble` 的动态切块模式。
3. 使用 `pypto.loop` 描述 kernel 内部循环。
4. 在 kernel 内使用静态条件、动态条件和循环边界条件。
5. 看懂 kernel 输入与输出的组织方式，以及它和普通算子/attention 的关系。

## 3. 这一节会依次练什么

| 练习场景 | 位置 | 关键点 |
| --- | --- | --- |
| 单动态 batch | 4.1 | 用 `pypto.DYNAMIC` 描述运行时 batch。 |
| 部分动态维度 | 4.2 | 只让 batch 动态，其余维度保持具体。 |
| 动态 Attention 预览 | 4.3 | 在四维 Q/K/V 上练习动态 batch 切块。 |
| 多个动态维度 | 4.4 | 同时处理 batch 和 hidden 两个动态维度。 |
| 基础 loop | 5.1 | 理解 `pypto.loop(start, stop, step)`。 |
| 编译期 print | 5.2 | 了解编译期打印的定位，不作为默认运行练习。 |
| loop 分块加标量 | 5.3 | 在 loop 中按 batch 分块执行 `x + y + val`。 |
| loop + dynamic axis | 5.4 | 用 `view + valid_shape` 处理动态轴和尾块。 |
| 嵌套 loop + if/else | 6.1 | 在循环内部根据索引选择不同分支。 |
| 静态 bool 条件 | 6.2 | 根据调用时配置选择计算路径。 |
| 动态索引条件 | 6.3 | 根据当前 loop 位置处理部分 tile。 |
| 循环边界条件 | 6.4 | 使用 `is_loop_begin` / `is_loop_end` 处理首尾 tile。 |
| 普通 OP 输入组织 | 7.1 | 观察多个输入和多个输出如何传入 kernel。 |
| Attention 输入组织 | 7.2 | 把同样的输入组织方式扩展到复杂模块。 |

练习数量较多时，阅读顺序比逐行记忆更重要：先理解动态切块，再理解循环，最后理解条件和输入组织。

### 3.1 建议运行顺序

建议按能力递进来运行这一节，而不是把每个单元当成彼此独立的小片段：

| 顺序 | 先看什么 | 重点问题 |
| --- | --- | --- |
| 1 | 动态 shape | 动态维度在哪里，真实 shape 如何进入 kernel？ |
| 2 | `view + valid_shape + assemble` | 局部 tile 怎么取，尾块怎么避免越界？ |
| 3 | `pypto.loop` | 循环是在 host 侧执行，还是进入 kernel 计算图？ |
| 4 | 条件分支 | 条件来自静态配置、循环索引，还是首尾边界？ |
| 5 | 输入组织 | 多个输入/输出如何按参数传递，并和 reference 对齐？ |

阅读时建议始终抓住三件事：动态维度在哪里、局部 tile 怎么取、结果写回哪里。

## 4. 动态 Shape

动态 shape 的核心问题是：kernel 编译时不知道某些维度的最终大小，但仍然需要写出稳定的计算逻辑。最常见的解决方式是：

```text
动态输入 -> 按固定 tile view 局部数据 -> 计算 -> assemble 写回输出
```

`valid_shape` 用来告诉 PyPTO 当前 tile 中有多少位置是真实有效的，这对最后一个不满 tile 的分块尤其重要。

### 阅读模板：动态 shape 的七步检查

动态 shape 代码看起来比静态代码长，主要是因为多了边界管理。可以固定按下面顺序读：

| 步骤 | 代码关键词 | 要回答的问题 |
| --- | --- | --- |
| 1. 声明动态维度 | `pypto.Tensor([pypto.DYNAMIC, ...])` | 哪些维度运行时才确定？ |
| 2. 读取真实维度 | `x.shape[0]`、`input_tensor.shape` | kernel 内如何拿到实际 batch / hidden / seq 长度？ |
| 3. 计算循环次数 | `(size + tile - 1) // tile` | 需要几个 tile 才能覆盖完整输入？ |
| 4. 计算当前 tile 边界 | `offset`、`end`、`(...).min(...)` | 当前循环处理哪一段？尾块是否越界？ |
| 5. 取局部视图 | `pypto.view(..., valid_shape=...)` | 当前 tile 的逻辑 shape 和真实有效区域是什么？ |
| 6. 局部计算 | `mul`、`add`、`softmax`、`attention_core` | 当前 tile 内部做什么数学计算？ |
| 7. 写回输出 | `pypto.assemble(...)` | 局部结果写回全局输出的哪个 offset？ |

只要这七步能对上，动态 batch、动态 hidden、动态 attention 本质上都是同一种结构。

### 4.1 单动态 batch 维

`dynamic_mul_kernel` 是最小动态 shape 模式。它只把第 0 维设成动态，隐藏维度仍然固定为 128。这样既保留了灵活性，也给编译器留下了足够明确的形状信息。

| 代码 | 含义 |
| --- | --- |
| `x: pypto.Tensor([pypto.DYNAMIC, 128], pypto.DT_FP16)` | 输入 batch 动态，hidden 固定为 128。 |
| `batch_size_dyn = x.shape[0]` | 运行时读取真实 batch。 |
| `b_loop = (batch_size_dyn + tile_b - 1) // tile_b` | 向上取整，确保最后一个不满 tile 的 batch 也被处理。 |
| `b_offset = idx * tile_b` | 当前 tile 的 batch 起点。 |
| `b_offset_end = (...).min(...)` | 当前 tile 的 batch 终点，不超过真实 batch。 |
| `valid_shape = [b_offset_end - b_offset, 128]` | 描述当前 tile 里真实有效的数据范围。 |
| `pypto.view(x, [tile_b, 128], [b_offset, 0], valid_shape=valid_shape)` | 从输入中取一个 batch tile。 |
| `result = pypto.mul(x_view, 2.0)` | 在局部 tile 上做逐元素乘 2。 |
| `pypto.assemble(result, [b_offset, 0], output)` | 把局部结果写回输出对应位置。 |

测试函数连续使用 batch size 8 和 16 调用同一个 kernel，目的是说明：同一段动态 kernel 可以接受不同 batch 的真实输入。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def dynamic_mul_kernel(
    x: pypto.Tensor([pypto.DYNAMIC, 128], pypto.DT_FP16),
    output: pypto.Tensor([pypto.DYNAMIC, 128], pypto.DT_FP16),
    tile_b: int):
    batch_size_dyn = x.shape[0]
    b_loop = (batch_size_dyn + tile_b - 1) // tile_b

    for idx in pypto.loop(b_loop):
        b_offset = idx * tile_b
        b_offset_end = (b_offset + tile_b).min(batch_size_dyn)
        valid_shape = [b_offset_end - b_offset, 128]
        x_view = pypto.view(x, [tile_b, 128], [b_offset, 0], valid_shape=valid_shape)
        pypto.set_vec_tile_shapes(1, 128)
        result = pypto.mul(x_view, 2.0)
        pypto.assemble(result, [b_offset, 0], output)


def test_dynamic_mul():
    device_local = get_device()
    for bs in [8, 16]:
        x = torch.randn(bs, 128, dtype=torch.float16, device=device_local)
        result = torch.zeros(bs, 128, dtype=torch.float16, device=device_local)
        dynamic_mul_kernel(x, result, bs)
        golden = x * 2.0
        print(f"dynamic_mul batch={bs}: {tuple(x.shape)} -> {tuple(result.shape)}")
        assert_allclose(to_numpy(result), to_numpy(golden), rtol=1e-3, atol=1e-3)


test_dynamic_mul()


### 4.2 部分动态维度

部分动态维度是更推荐的写法：只把真正会变的 batch 设成动态，`seqlen / head / dim` 仍然从输入 shape 中读取并保持清晰。这样写出来的 kernel 通常更容易优化，也更容易阅读。

本例输入 shape 是 `[B, S, H, D]`，其中只有 `B` 动态：

```text
input_tensor:  [B, 32, 1, 256]
input_view:    [1, 32, 1, 256]
softmax_out:   [1, 32, 1, 256]
output_tensor: [B, 32, 1, 256]
```

`pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32)` 中的 `...` 表示除第 0 维之外的其它维度从运行时输入继承。这里使用切片 `input_tensor[b_offset:b_offset_end, :seqlen, :head, :dim]`，没有显式 `valid_shape`，因为 `tile_b=1` 且测试 batch 都能整齐覆盖。

In [ ]:
def softmax_core(input_tensor: pypto.Tensor) -> pypto.Tensor:
    return pypto.softmax(input_tensor, dim=-1)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def dynamic_softmax_kernel(
    input_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32)):
    tile_b = 1
    bs_dyn, seqlen, head, dim = input_tensor.shape
    b_loop = bs_dyn // tile_b

    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    for idx in pypto.loop(0, b_loop, 1, name="LOOP_L0_bIdx", idx_name="idx"):
        b_offset = idx * tile_b
        b_offset_end = (idx + 1) * tile_b
        input_view = input_tensor[b_offset:b_offset_end, :seqlen, :head, :dim]
        softmax_out = softmax_core(input_view)
        output_tensor[b_offset:b_offset_end, ...] = softmax_out


def test_dynamic_partial():
    device_local = get_device()
    seqlen, head, dim = 32, 1, 256
    for bs in [8, 32]:
        shape = (bs, seqlen, head, dim)
        x = torch.rand(shape, dtype=torch.float32, device=device_local)
        y = torch.zeros(shape, dtype=torch.float32, device=device_local)
        dynamic_softmax_kernel(x, y)
        golden = torch.softmax(x, dim=-1)
        print(f"dynamic_partial batch={bs}: {tuple(x.shape)} -> {tuple(y.shape)}")
        assert_allclose(to_numpy(y), to_numpy(golden), rtol=1e-3, atol=1e-3)


test_dynamic_partial()


### 4.3 动态 Attention 预览

这里的 attention 练习主要是为了展示动态 batch 与四维张量切块，不在本节深讲 Attention 数学。真正的 Attention 结构会放到 4.5 继续展开。

此处只需要抓住一点：`q / k / v` 都沿 batch 维做 `view`，每个 tile 内部再执行 `Q @ K^T -> softmax -> @ V`。

| 张量 | shape 含义 |
| --- | --- |
| `q` | `[B, num_heads, seq_len_q, head_dim]` |
| `k` | `[B, num_heads, seq_len_kv, head_dim]` |
| `v` | `[B, num_heads, seq_len_kv, head_dim]` |
| `q_view / k_view / v_view` | `[tile, num_heads, seq_len, head_dim]`，只包含当前 batch tile。 |
| `res` | `[tile, num_heads, seq_len_q, head_dim]` |
| `output_tensor` | `[B, num_heads, seq_len_q, head_dim]` |

这段代码和 4.5 的 Attention 主线高度相关。本节先不深挖 `transpose` 和 attention score，只关注动态 batch 如何包住一个复杂模块。

In [ ]:
@dataclass
class AttentionConfig:
    num_heads: int = 8
    head_dim: int = 64
    scale: Optional[float] = None
    dtype: pypto.DataType = pypto.DT_FP32


def scaled_dot_product_attention_golden(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                                        scale: float, attn_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    scores = torch.matmul(q, k.transpose(-2, -1))
    scores = scores * scale
    if attn_mask is not None:
        scores = scores + attn_mask
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, v)


def scaled_dot_product_attention_core(q: pypto.Tensor, k: pypto.Tensor, v: pypto.Tensor,
                                      scale: float, dtype: pypto.DataType) -> pypto.Tensor:
    k_t = pypto.transpose(k, 2, 3)
    scores = pypto.matmul(q, k_t, out_dtype=dtype)
    scores_scaled = scores * scale
    attn_weights = pypto.softmax(scores_scaled, dim=-1)
    return pypto.matmul(attn_weights, v, out_dtype=dtype)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def dynamic_attention_kernel(
    q: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    k: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    v: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    config: AttentionConfig,
    tile: int):
    bs_dyn = q.shape[0]
    head = config.num_heads
    dim = config.head_dim
    q_len = q.shape[2]
    kv_len = k.shape[2]
    scale = config.scale if config.scale is not None else (1.0 / (dim ** 0.5))
    cube_tiling = 64
    pypto.set_cube_tile_shapes([cube_tiling, cube_tiling], [cube_tiling, cube_tiling], [cube_tiling, cube_tiling])
    bs_loop = (bs_dyn + tile - 1) // tile

    for bss_idx in pypto.loop(bs_loop):
        bs_offset = bss_idx * tile
        bs_offset_end = (bs_offset + tile).min(bs_dyn)
        q_view = pypto.view(q, [tile, head, q_len, dim], [bs_offset, 0, 0, 0], valid_shape=[bs_offset_end - bs_offset, head, q_len, dim])
        k_view = pypto.view(k, [tile, head, kv_len, dim], [bs_offset, 0, 0, 0], valid_shape=[bs_offset_end - bs_offset, head, kv_len, dim])
        v_view = pypto.view(v, [tile, head, kv_len, dim], [bs_offset, 0, 0, 0], valid_shape=[bs_offset_end - bs_offset, head, kv_len, dim])
        pypto.set_vec_tile_shapes(1, 8, 16, 64)
        res = scaled_dot_product_attention_core(q_view, k_view, v_view, scale, config.dtype)
        pypto.assemble(res, [bs_offset, 0, 0, 0], output_tensor)


def test_dynamic_attention():
    device_local = get_device()
    num_heads, head_dim = 8, 64
    config = AttentionConfig(num_heads=num_heads, head_dim=head_dim, dtype=pypto.DT_FP32)
    test_cases = [(2, 16, 16), (4, 32, 32), (8, 64, 64)]
    for batch_size, seq_len_q, seq_len_kv in test_cases:
        q = torch.randn(batch_size, num_heads, seq_len_q, head_dim, dtype=torch.float32, device=device_local)
        k = torch.randn(batch_size, num_heads, seq_len_kv, head_dim, dtype=torch.float32, device=device_local)
        v = torch.randn(batch_size, num_heads, seq_len_kv, head_dim, dtype=torch.float32, device=device_local)
        out = torch.empty(batch_size, num_heads, seq_len_q, head_dim, dtype=torch.float32, device=device_local)
        dynamic_attention_kernel(q, k, v, out, config, batch_size)
        golden = scaled_dot_product_attention_golden(q, k, v, 1.0 / (head_dim ** 0.5)).cpu()
        print(f"dynamic_attention batch={batch_size}, seq_q={seq_len_q}, seq_kv={seq_len_kv}")
        assert_allclose(to_numpy(out), to_numpy(golden), rtol=3e-3, atol=3e-3)


test_dynamic_attention()

### 4.4 多个动态维度

多个动态维度的写法是在 batch 维循环的基础上，再套一层 hidden 维循环。代码看起来长一点，但仍然是同一个模式：每一层动态维度都需要计算 offset、end 和 valid shape。

| 层级 | 关键变量 | 作用 |
| --- | --- | --- |
| batch 维 | `b_offset`、`b_offset_end`、`valid_b` | 决定当前处理哪些行。 |
| hidden 维 | `h_offset`、`h_offset_end`、`valid_h` | 决定当前处理哪些列。 |
| view | `pypto.view(x, [tile_b, tile_h], [b_offset, h_offset], ...)` | 取二维 tile。 |
| compute | `pypto.add(x_view, y_view)` | 对当前二维 tile 做加法。 |
| assemble | `pypto.assemble(result, [b_offset, h_offset], output)` | 写回二维输出的对应区域。 |

当只有 batch 动态时，一层 loop 就够；当 batch 和 hidden 都动态时，就需要嵌套 loop。后续如果 seq_len 也动态，道理仍然类似，只是边界管理会更复杂。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def dynamic_add_kernel(
    x: pypto.Tensor([pypto.DYNAMIC, pypto.DYNAMIC], pypto.DT_FP16),
    y: pypto.Tensor([pypto.DYNAMIC, pypto.DYNAMIC], pypto.DT_FP16),
    output: pypto.Tensor([pypto.DYNAMIC, pypto.DYNAMIC], pypto.DT_FP16),
    tile_b: int,
    tile_h: int):
    batch_dyn = x.shape[0]
    hidden_dyn = x.shape[1]
    b_loop = (batch_dyn + tile_b - 1) // tile_b

    for b_idx in pypto.loop(b_loop):
        b_offset = b_idx * tile_b
        b_offset_end = (b_offset + tile_b).min(batch_dyn)
        valid_b = b_offset_end - b_offset
        h_loop = (hidden_dyn + tile_h - 1) // tile_h
        for h_idx in pypto.loop(h_loop):
            h_offset = h_idx * tile_h
            h_offset_end = (h_offset + tile_h).min(hidden_dyn)
            valid_h = h_offset_end - h_offset
            x_view = pypto.view(x, [tile_b, tile_h], [b_offset, h_offset], valid_shape=[valid_b, valid_h])
            y_view = pypto.view(y, [tile_b, tile_h], [b_offset, h_offset], valid_shape=[valid_b, valid_h])
            pypto.set_vec_tile_shapes(tile_b, tile_h)
            result = pypto.add(x_view, y_view)
            pypto.assemble(result, [b_offset, h_offset], output)


def test_dynamic_multi_dim():
    device_local = get_device()
    for bs, hs in [(8, 64), (16, 128)]:
        x = torch.randn(bs, hs, dtype=torch.float16, device=device_local)
        y = torch.randn(bs, hs, dtype=torch.float16, device=device_local)
        result = torch.zeros(bs, hs, dtype=torch.float16, device=device_local)
        dynamic_add_kernel(x, y, result, bs, hs)
        golden = x + y
        print(f"dynamic_multi_dim batch={bs}, hidden={hs}")
        assert_allclose(to_numpy(result), to_numpy(golden), rtol=1e-3, atol=1e-3)


test_dynamic_multi_dim()

## 5. Loop 控制

`pypto.loop` 描述的是进入计算图的循环，不是普通 Python for 循环的简单替代。它常见于分块处理、动态 batch、编译期展开和局部调试。

| 对比项 | Python `for` | `pypto.loop` |
| --- | --- | --- |
| 运行层次 | host 侧 Python 解释执行 | PyPTO kernel / 计算图内部描述 |
| 常见用途 | 准备数据、调用测试、多次运行 kernel | 设备侧分块、动态 batch、控制流生成 |
| 索引含义 | 普通 Python int | 可进入 PyPTO 图的循环索引 |
| 是否可参与 kernel 内条件 | 不在 kernel 图内 | 可以，例如 `if idx < 2`、`is_loop_begin(idx)` |

所以看到 `pypto.loop` 时，不要只把它当成语法糖。它是在告诉 PyPTO：这个循环本身是 kernel 逻辑的一部分。

### 5.1 基础 Loop

`loop_basic_kernel` 展示了 `start / stop / step` 的写法。两个循环得到同样的数学结果，只是分块步长不同。

| 循环 | step | 每次处理的行范围 | 写入输出 |
| --- | --- | --- | --- |
| 第一个 loop | `1` | `[bs_idx * s : (bs_idx + 1) * s]` | `out0` |
| 第二个 loop | `2` | `[bs_idx * s : (bs_idx + 2) * s]` | `out1` |

测试里 `s=64`、`n=8`，完整输入 shape 是 `[512, 64]`。两个输出都应该等于 `input_t1 + input_t2`，这说明改变 loop step 只改变分块组织，不改变最终数学结果。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def loop_basic_kernel(t0: pypto.Tensor(), t1: pypto.Tensor(), out0: pypto.Tensor(), out1: pypto.Tensor(), s: int, n: int):
    pypto.set_vec_tile_shapes(64, 64)
    for bs_idx in pypto.loop(0, n, 1):
        t0s = t0[bs_idx * s: (bs_idx + 1) * s, :]
        t1s = t1[bs_idx * s: (bs_idx + 1) * s, :]
        out0[bs_idx * s: (bs_idx + 1) * s, :] = pypto.add(t0s, t1s)
    new_step = 2
    for bs_idx in pypto.loop(0, n, new_step):
        t0s = t0[bs_idx * s: (bs_idx + new_step) * s, :]
        t1s = t1[bs_idx * s: (bs_idx + new_step) * s, :]
        out1[bs_idx * s: (bs_idx + new_step) * s, :] = pypto.add(t0s, t1s)


def test_loop_basic() -> None:
    device_local = get_device()
    s, n = 64, 8
    shape = (n * s, s)
    input_t1 = torch.randn(shape, dtype=torch.float16, device=device_local)
    input_t2 = torch.randn(shape, dtype=torch.float16, device=device_local)
    output1 = torch.empty(shape, dtype=torch.float16, device=device_local)
    output2 = torch.empty(shape, dtype=torch.float16, device=device_local)
    loop_basic_kernel(input_t1, input_t2, output1, output2, s, n)
    expected = input_t1 + input_t2
    print(f"loop_basic shape: {shape}")
    assert (output1 - expected).abs().max().item() < 1e-2
    assert (output2 - expected).abs().max().item() < 1e-2


test_loop_basic()

### 5.2 编译期 print

这一段的重点不是结果计算，而是提醒：kernel 内部的 `print` 主要发生在编译期，用来观察图生成和循环展开。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def loop_compile_phase_print_kernel(
    in_t0: pypto.Tensor(), in_t1: pypto.Tensor(), out_t0: pypto.Tensor(), out_t1: pypto.Tensor()
):
    pypto.set_vec_tile_shapes(64, 64)
    note = '''
    Below are demonstrations of print usage within loops.
    It executes only during compilation, cannot truly print variable values,
    and the number of prints is related to the number of subgraphs generated.
    '''
    separator = "*" * 60
    print(note)
    print(separator)
    cnt_inside_cond = 0
    cnt_outside_cond = 0
    for outside_idx in pypto.loop(5):
        print(f"outside_idx: {outside_idx}")
        for inside_idx in pypto.loop(3):
            print(f"inside_idx: {outside_idx}")
            res = pypto.add(in_t0, in_t0)
            print(f"res: {res}")
            if outside_idx < 3:
                print(f"(outside_idx < 3)_count: {cnt_outside_cond}")
                cnt_outside_cond = cnt_outside_cond + 1
                res = pypto.add(in_t0, in_t0)
            else:
                res = pypto.sub(in_t0, in_t0)
            if inside_idx < 2:
                print(f"(inside_idx < 2)_count: {cnt_inside_cond}")
                cnt_inside_cond = cnt_inside_cond + 1
                res = pypto.div(in_t0, in_t0)
            else:
                res = pypto.add(in_t1, in_t1)
            out_t0.move(pypto.add(in_t0, in_t0))
            out_t1.move(pypto.add(in_t1, in_t1))
    print(separator)


def test_loop_compile_phase_print() -> None:
    """Test loop compile phase print"""
    print("=" * 60)
    print("Test: Loop Compile Phase Print Feature")
    print("=" * 60)

    device = get_device()

    m, n = 6, 8
    shape = (m, n)
    input_t1 = torch.randn(shape, dtype=torch.float16, device=device)
    input_t2 = torch.randn(shape, dtype=torch.float16, device=device)
    output_t1 = torch.empty(shape, dtype=torch.float16, device=device)
    output_t2 = torch.empty(shape, dtype=torch.float16, device=device)
    loop_compile_phase_print_kernel(input_t1, input_t2, output_t1, output_t2)
    expected_t1 = input_t1 + input_t1
    expected_t2 = input_t2 + input_t2
    max_diff_t1 = (output_t1 - expected_t1).abs().max().item()
    max_diff_t2 = (output_t2 - expected_t2).abs().max().item()
    print(f"Max difference from PyTorch: {max_diff_t1:.6f}")
    print(f"Max difference from PyTorch: {max_diff_t2:.6f}")
    assert max_diff_t1 < 1e-2, "Result mismatch!"
    assert max_diff_t2 < 1e-2, "Result mismatch!"
    print("✓ Test loop compile phase print completed successfully")
    print()

test_loop_compile_phase_print()

### 5.3 Loop 分块加标量

`add_kernel` 展示的是最直接的 loop 分块写法：每次取一个 batch tile，计算 `x + y + val`，再用 `pypto.assemble` 写回输出。这里使用普通切片取 tile，适合 batch 能整齐覆盖的简单场景。代码在下一小节展示。

| 代码 | 含义 |
| --- | --- |
| `b = input0.shape[0]` | 读取运行时 batch。 |
| `for idx in pypto.loop(b_loop)` | 在 kernel 内按 batch tile 循环。 |
| `input0[b_offset:b_offset_end, ...]` | 用普通切片取当前 tile。 |
| `t0_sub + t1_sub + val` | 在局部 tile 上完成逐元素加法。 |
| `pypto.assemble(...)` | 把局部结果写回完整输出。 |


### 5.4 Loop 与动态轴

`add_scalar_loop_dynamic_axis_kernel` 做的数学结果仍然是 `x + y + val`，但它显式使用 `view` 和 `valid_shape`。当 batch 维真的会变化时，这种写法更稳，也更接近动态 shape 的推荐模式。

| 练习 | 取 tile 方式 | 写回方式 | 适合场景 |
| --- | --- | --- | --- |
| `add_kernel` | `input0[b_offset:b_offset_end, ...]` 普通切片 | `pypto.assemble(...)` | batch 可以整齐按 `tile_b` 处理的简单场景。 |
| `add_scalar_loop_dynamic_axis_kernel` | `pypto.view(..., valid_shape=...)` | `pypto.assemble(...)` | batch 可能变化，尾块可能不满 tile 的推荐写法。 |

验证时 PyTorch reference 都是 `golden = x + y + val`。如果结果不一致，优先检查 `b_offset`、`b_offset_end` 和 `valid_shape` 是否对齐。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_kernel(
    input0: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    input1: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    val: int):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    b = input0.shape[0]
    tile_b = 1
    b_loop = b // tile_b
    for idx in pypto.loop(b_loop):
        b_offset = idx * tile_b
        b_offset_end = (idx + 1) * tile_b
        t0_sub = input0[b_offset:b_offset_end, ...]
        t1_sub = input1[b_offset:b_offset_end, ...]
        t3_sub = t0_sub + t1_sub + val
        pypto.assemble(t3_sub, [b_offset, 0, 0, 0], output)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_scalar_loop_dynamic_axis_kernel(
    input0: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    input1: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    val: int):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    b, w, n, c = input0.shape
    tile_b = 1
    b_loop = b // tile_b
    for idx in pypto.loop(b_loop):
        b_offset = idx * tile_b
        b_offset_end = min((idx + 1) * tile_b, b)
        valid_shape = [b_offset_end - b_offset, w, n, c]
        t0_sub = pypto.view(input0, [tile_b, w, n, c], [b_offset, 0, 0, 0], valid_shape=valid_shape)
        t1_sub = pypto.view(input1, [tile_b, w, n, c], [b_offset, 0, 0, 0], valid_shape=valid_shape)
        t3_sub = t0_sub + t1_sub + val
        pypto.assemble(t3_sub, [b_offset, 0, 0, 0], output)


def test_add_scalar_loop() -> None:
    device_local = get_device()
    shape = (32, 32, 1, 256)
    val = 1
    x = torch.rand(shape, dtype=torch.float32, device=device_local)
    y = torch.rand(shape, dtype=torch.float32, device=device_local)
    z = torch.empty(shape, dtype=torch.float32, device=device_local)
    add_kernel(x, y, z, val)
    golden = x + y + val
    print(f"add_scalar_loop shape: {shape}")
    assert_allclose(to_numpy(z), to_numpy(golden), rtol=3e-3, atol=3e-3)


def test_add_scalar_loop_dyn_axis() -> None:
    device_local = get_device()
    shape = (32, 32, 1, 256)
    val = 1
    input_data0 = torch.rand(shape, dtype=torch.float32, device=device_local)
    input_data1 = torch.rand(shape, dtype=torch.float32, device=device_local)
    output_data = torch.empty(shape, dtype=torch.float32, device=device_local)
    add_scalar_loop_dynamic_axis_kernel(input_data0, input_data1, output_data, val)
    golden = input_data0 + input_data1 + val
    print(f"add_scalar_loop_dyn_axis shape: {shape}")
    assert_allclose(to_numpy(output_data), to_numpy(golden), rtol=3e-3, atol=3e-3)


test_add_scalar_loop()
test_add_scalar_loop_dyn_axis()

## 6. 条件分支

条件分支让 kernel 能够根据静态参数、循环索引或循环边界执行不同路径。这里重点练习四个模式：嵌套条件、静态 bool 条件、动态索引条件和循环边界条件。

阅读条件分支时，先判断条件来自哪里：

| 条件来源 | 代表写法 | 直观含义 |
| --- | --- | --- |
| 循环索引 | `if i == 0` | 当前 loop 走到哪一层或哪一块。 |
| host 静态参数 | `if flag` | 调用 kernel 时传入的配置开关。 |
| 动态 loop 索引 | `if idx < 2` | 只对前几个 tile 做特殊处理。 |
| loop 边界 | `pypto.is_loop_begin(idx)`、`pypto.is_loop_end(idx)` | 只处理首块或尾块。 |

### 6.1 嵌套 loop + if/else

`nested_loops_with_conditions_kernel` 在二维循环里根据 `i == 0` 选择加法或减法。它适合先观察“循环索引进入条件分支”这件事：第一行走 `a + b`，第二行走 `a - b`。

### 6.2 静态 bool 条件

`add_scalar_loop_dyn_axis_static_cond_kernel_static` 和 `add_scalar_loop_dyn_axis_static_cond_kernel_dynamic` 都把 `flag` 作为 host 侧传入的配置开关。`flag=False` 时只计算 `x + y`，`flag=True` 时计算 `x + y + val`。

### 6.3 动态索引条件

`add_scalar_loop_dyn_axis_dyn_cond_kernel` 根据当前 loop 索引选择路径：前两个 batch tile 额外加 `val`，其余 tile 保持 `x + y`。这种模式适合根据 tile 位置做局部特殊处理。

### 6.4 循环边界条件

`add_scalar_loop_dyn_axis_dyn_loop_cond_kernel` 使用 `pypto.is_loop_begin(idx)` 和 `pypto.is_loop_end(idx)` 识别首尾 tile。这里第一个 batch 加 `val`，最后一个 batch 加 `val + 1`，中间 batch 只做 `x + y`。

| 练习 | reference 逻辑 |
| --- | --- |
| `test_nested_loops_with_conditions` | `golden[0] = a[0] + b[0]`?`golden[1] = a[1] - b[1]`? |
| `test_add_scalar_loop_dyn_axis_static_cond` | `flag=False` 时是 `x + y`，`flag=True` 时是 `x + y + val`。 |
| `test_add_scalar_loop_dynamic_axis_dynamic_cond` | 先整体 `x + y`，再只给前两个 batch 加 `val`。 |
| `test_add_scalar_loop_dynamic_axis_dynamic_loop_cond` | 第一个 batch 加 `val`，最后一个 batch 加 `val + 1`。 |

写 condition kernel 时，reference 一定要把分支行为写出来，而不是只比较一个笼统的 `x + y`。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def nested_loops_with_conditions_kernel(
    a: pypto.Tensor([pypto.DYNAMIC, pypto.DYNAMIC], pypto.DT_FP32),
    b: pypto.Tensor([pypto.DYNAMIC, pypto.DYNAMIC], pypto.DT_FP32),
    y: pypto.Tensor([pypto.DYNAMIC, pypto.DYNAMIC], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    for i in pypto.loop(2):
        for j in pypto.loop(2):
            a_view = a[i:i + 1, j:j + 1]
            b_view = b[i:i + 1, j:j + 1]
            if i == 0:
                y[i:i + 1, j:j + 1] = a_view + b_view
            else:
                y[i:i + 1, j:j + 1] = a_view - b_view


def add_core(input0: pypto.Tensor, input1: pypto.Tensor, output: pypto.Tensor, val: int, add1_flag: bool = False):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    b = input0.shape[0]
    tile_b = 1
    b_loop = b // tile_b
    for idx in pypto.loop(b_loop):
        b_offset = idx * tile_b
        b_offset_end = (idx + 1) * tile_b
        t0_sub = input0[b_offset:b_offset_end, ...]
        t1_sub = input1[b_offset:b_offset_end, ...]
        t3_sub = t0_sub + t1_sub
        if add1_flag:
            output[b_offset:b_offset_end, ...] = t3_sub + val
        else:
            output[b_offset:b_offset_end, ...] = t3_sub


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_scalar_loop_dyn_axis_static_cond_kernel_static(input0: pypto.Tensor(), input1: pypto.Tensor(),
                                                       output: pypto.Tensor(), val: int, flag: bool):
    add_core(input0, input1, output, val, flag)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_scalar_loop_dyn_axis_static_cond_kernel_dynamic(
    input0: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    input1: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    val: int,
    flag: bool):
    add_core(input0, input1, output, val, flag)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_scalar_loop_dyn_axis_dyn_cond_kernel(
    input0: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    input1: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    val: int):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    b = input0.shape[0]
    for idx in pypto.loop(b):
        t3_sub = input0[idx:idx + 1, ...] + input1[idx:idx + 1, ...]
        if idx < 2:
            output[idx:idx + 1, ...] = t3_sub + val
        else:
            output[idx:idx + 1, ...] = t3_sub


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_scalar_loop_dyn_axis_dyn_loop_cond_kernel(
    input0: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    input1: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    val: int):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    b = input0.shape[0]
    for idx in pypto.loop(b):
        t3_sub = input0[idx:idx + 1, ...] + input1[idx:idx + 1, ...]
        if pypto.is_loop_begin(idx):
            output[idx:idx + 1, ...] = t3_sub + val
        elif pypto.is_loop_end(idx):
            output[idx:idx + 1, ...] = t3_sub + val + 1
        else:
            output[idx:idx + 1, ...] = t3_sub


In [ ]:
def test_nested_loops_with_conditions() -> None:
    device_local = get_device()
    shape = (2, 2)
    a = torch.rand(shape, dtype=torch.float32, device=device_local)
    b = torch.rand(shape, dtype=torch.float32, device=device_local)
    y = torch.zeros(shape, dtype=torch.float32, device=device_local)
    nested_loops_with_conditions_kernel(a, b, y)
    golden = torch.zeros(shape, dtype=torch.float32, device=device_local)
    golden[0] = a[0] + b[0]
    golden[1] = a[1] - b[1]
    print("nested_loops_with_conditions shape:", shape)
    assert_allclose(to_numpy(y), to_numpy(golden), rtol=1e-3, atol=1e-3)


def test_add_scalar_loop_dyn_axis_static_cond() -> None:
    device_local = get_device()
    shape = (32, 32, 1, 256)
    val = 1
    input_data0 = torch.rand(shape, dtype=torch.float32, device=device_local)
    input_data1 = torch.rand(shape, dtype=torch.float32, device=device_local)
    output_data = torch.empty(shape, dtype=torch.float32, device=device_local)
    add_scalar_loop_dyn_axis_static_cond_kernel_static(input_data0, input_data1, output_data, val, False)
    golden = input_data0 + input_data1
    output_data2 = torch.empty(shape, dtype=torch.float32, device=device_local)
    add_scalar_loop_dyn_axis_static_cond_kernel_dynamic(input_data0, input_data1, output_data2, val, True)
    golden2 = input_data0 + input_data1 + val
    print("static_cond shape:", shape)
    assert_allclose(to_numpy(output_data), to_numpy(golden), rtol=3e-3, atol=3e-3)
    assert_allclose(to_numpy(output_data2), to_numpy(golden2), rtol=3e-3, atol=3e-3)


def test_add_scalar_loop_dynamic_axis_dynamic_cond() -> None:
    device_local = get_device()
    shape = (32, 32, 1, 256)
    val = 1
    input_data0 = torch.rand(shape, dtype=torch.float32, device=device_local)
    input_data1 = torch.rand(shape, dtype=torch.float32, device=device_local)
    output_data = torch.empty(shape, dtype=torch.float32, device=device_local)
    add_scalar_loop_dyn_axis_dyn_cond_kernel(input_data0, input_data1, output_data, val)
    golden = input_data0 + input_data1
    golden[0:2, ...] = golden[0:2, ...] + val
    print("dynamic_cond shape:", shape)
    assert_allclose(to_numpy(output_data), to_numpy(golden), rtol=3e-3, atol=3e-3)


def test_add_scalar_loop_dynamic_axis_dynamic_loop_cond() -> None:
    device_local = get_device()
    shape = (32, 32, 1, 256)
    val = 1
    input_data0 = torch.rand(shape, dtype=torch.float32, device=device_local)
    input_data1 = torch.rand(shape, dtype=torch.float32, device=device_local)
    output_data = torch.empty(shape, dtype=torch.float32, device=device_local)
    add_scalar_loop_dyn_axis_dyn_loop_cond_kernel(input_data0, input_data1, output_data, val)
    golden = input_data0 + input_data1
    golden[0:1, ...] = golden[0:1, ...] + val
    golden[31:32, ...] = golden[31:32, ...] + val + 1
    print("loop_boundary_cond shape:", shape)
    assert_allclose(to_numpy(output_data), to_numpy(golden), rtol=3e-3, atol=3e-3)


In [ ]:
test_nested_loops_with_conditions()
test_add_scalar_loop_dyn_axis_static_cond()
test_add_scalar_loop_dynamic_axis_dynamic_cond()
test_add_scalar_loop_dynamic_axis_dynamic_loop_cond()


## 7. Kernel 输入顺序

这一部分关注 kernel 输入与输出的组织方式。先看一个普通 elementwise OP，再看一个 attention OP：普通 OP 更容易看懂，attention OP 则说明同样的组织方式可以扩展到更复杂的模块。

### 7.1 普通 OP 输入组织

`op_unordered_input_kernel` 用 `out1 = a + b` 和 `out2 = a * b` 展示多个输入、多个输出如何组织。重点是看清 kernel 参数里哪些是输入，哪些是由 kernel 写回的输出。

### 7.2 Attention 输入组织

`unordered_input_attention_kernel` 把同样的输入组织方式扩展到 Attention。它复用 `scaled_dot_product_attention_core`，但不在 JIT kernel 内嵌套调用另一个 JIT kernel；这样既保留动态 batch 的切块逻辑，也避免记录阶段把包装后的 kernel 当成普通输入处理。

这里不再展开 Attention 数学，后续 4.5 会专门讲。

| 练习 | 输入 | 输出 | reference |
| --- | --- | --- | --- |
| 普通 OP | `a`、`b` | `out1`、`out2` | `out1 = a + b`，`out2 = a * b` |
| Attention | `q`?`k`?`v`?`config`?`tile` | `out` | `scaled_dot_product_attention_golden(q, k, v, scale)` |

这一小节的学习重点是 kernel 输入和输出如何组织，不是重新讲 Attention。复杂 kernel 仍然可以按“声明输入 -> 调用 core -> 写回输出 -> reference 验证”的顺序阅读。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def op_unordered_input_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out1: pypto.Tensor([], pypto.DT_FP32),
    out2: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(16, 16)
    out1.move(a + b)
    out2.move(a * b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def unordered_input_attention_kernel(
    q: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    k: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    v: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    config: AttentionConfig,
    tile: int):
    bs_dyn = q.shape[0]
    head = config.num_heads
    dim = config.head_dim
    q_len = q.shape[2]
    kv_len = k.shape[2]
    scale = config.scale if config.scale is not None else (1.0 / (dim ** 0.5))
    cube_tiling = 64
    pypto.set_cube_tile_shapes([cube_tiling, cube_tiling], [cube_tiling, cube_tiling], [cube_tiling, cube_tiling])
    bs_loop = (bs_dyn + tile - 1) // tile

    for bs_idx in pypto.loop(bs_loop):
        bs_offset = bs_idx * tile
        bs_offset_end = (bs_offset + tile).min(bs_dyn)
        q_view = pypto.view(q, [tile, head, q_len, dim], [bs_offset, 0, 0, 0],
                            valid_shape=[bs_offset_end - bs_offset, head, q_len, dim])
        k_view = pypto.view(k, [tile, head, kv_len, dim], [bs_offset, 0, 0, 0],
                            valid_shape=[bs_offset_end - bs_offset, head, kv_len, dim])
        v_view = pypto.view(v, [tile, head, kv_len, dim], [bs_offset, 0, 0, 0],
                            valid_shape=[bs_offset_end - bs_offset, head, kv_len, dim])
        pypto.set_vec_tile_shapes(1, 8, 16, 64)
        res = scaled_dot_product_attention_core(q_view, k_view, v_view, scale, config.dtype)
        pypto.assemble(res, [bs_offset, 0, 0, 0], output_tensor)


def test_unordered_input_op() -> None:
    device_local = get_device()
    shape = (3, 2)
    a = torch.rand(shape, dtype=torch.float32, device=device_local)
    b = torch.rand(shape, dtype=torch.float32, device=device_local)
    y1 = torch.empty(shape, dtype=torch.float32, device=device_local)
    y2 = torch.empty(shape, dtype=torch.float32, device=device_local)
    op_unordered_input_kernel(a, b, y1, y2)
    golden1 = torch.add(a, b).cpu()
    golden2 = torch.mul(a, b).cpu()
    print("unordered_input_op shape:", shape)
    assert_allclose(to_numpy(y1), to_numpy(golden1), rtol=1e-3, atol=1e-3)
    assert_allclose(to_numpy(y2), to_numpy(golden2), rtol=1e-3, atol=1e-3)


def test_unordered_input_attention() -> None:
    device_local = get_device()
    batch_size, num_heads, seq_len_q, seq_len_kv, head_dim = 8, 8, 64, 64, 64
    q = torch.randn(batch_size, num_heads, seq_len_q, head_dim, dtype=torch.float32, device=device_local)
    k = torch.randn(batch_size, num_heads, seq_len_kv, head_dim, dtype=torch.float32, device=device_local)
    v = torch.randn(batch_size, num_heads, seq_len_kv, head_dim, dtype=torch.float32, device=device_local)
    out = torch.empty(batch_size, num_heads, seq_len_q, head_dim, dtype=torch.float32, device=device_local)
    config = AttentionConfig(num_heads=num_heads, head_dim=head_dim, dtype=pypto.DT_FP32)
    unordered_input_attention_kernel(q, k, v, out, config, batch_size)
    golden = scaled_dot_product_attention_golden(q, k, v, 1.0 / (head_dim ** 0.5))
    print("unordered_input_attention shape:", tuple(q.shape), "->", tuple(out.shape))
    assert_allclose(to_numpy(out), to_numpy(golden), rtol=3e-3, atol=3e-3)


In [ ]:
test_unordered_input_op()
test_unordered_input_attention()


## 8. 本节 API 速览

| API | 作用 |
| --- | --- |
| `pypto.DYNAMIC` | 声明动态维度 |
| `pypto.view` | 从动态输入中取局部 tile |
| `valid_shape` | 描述当前 tile 的有效区域 |
| `pypto.assemble` | 把局部计算结果写回大 Tensor |
| `pypto.loop` | 描述计算图内循环 |
| `pypto.is_loop_begin` | 判断当前循环是否在起点 |
| `pypto.is_loop_end` | 判断当前循环是否在终点 |
| `pypto.set_vec_tile_shapes` | 配置向量类算子的 tile |
| `pypto.set_cube_tile_shapes` | 配置矩阵类算子的 tile |

把这些 API 串起来，基本就能读懂 PyPTO 中级 runtime 代码。

最常见的组合方式有三种：

| 组合 | 典型用途 |
| --- | --- |
| `DYNAMIC + loop + view + valid_shape + assemble` | 动态 batch 或动态 hidden 的分块处理。 |
| `loop + if idx ...` | 根据循环位置执行不同计算。 |
| `loop + is_loop_begin / is_loop_end` | 对首块、尾块做特殊边界处理。 |

## 9. 课后练习

本节练习用于复盘动态 shape、loop、condition 和输入组织。请结合动态切块、循环边界和条件分支代码完成以下题目。

1. （选择题）为什么动态 shape 代码通常要写 `view -> compute -> assemble`？  
   A. 为了按固定 tile 取局部数据，计算后写回全局输出  
   B. 为了让所有维度都变成常量  
   C. 为了删除输出 Tensor  
   D. 为了跳过 kernel 编译
2. （选择题）只把 batch 设成动态，相比所有维度都动态，主要优势是什么？  
   A. 保留更多静态 shape 信息，更容易优化和阅读  
   B. 让所有 Tensor 自动变成标量  
   C. 禁止使用 `pypto.loop`  
   D. 必须关闭验证
3. （选择题）`pypto.loop` 和普通 Python `for` 的关键区别是什么？  
   A. `pypto.loop` 是 kernel 计算图的一部分，普通 Python `for` 在 Host 侧执行  
   B. `pypto.loop` 只能打印日志  
   C. 普通 Python `for` 可以直接变成设备侧循环  
   D. 二者没有区别
4. （填空题）静态 bool 条件适合________；动态循环索引条件适合________。
5. （填空题）`is_loop_begin` 和 `is_loop_end` 常用于处理________问题。

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/04.04_answer.txt


## 10. 本节小结

到这里，你已经练习了动态 shape、loop、condition 和输入组织。它们看起来分散，但核心都服务于同一个目标：让 PyPTO kernel 能够描述真实场景里的动态执行逻辑。

下一节会进入 Attention 与 Transformer 组合，把这里的动态 batch、loop、transpose、matmul 和 softmax 进一步组织成完整结构。